In [2]:
import sqlite3
import pandas as pd
import os
from mappings import resolve_company_sec, fetch_sec_tickers, get_biotech_universe, clean_name
from scraper import fetch_trials
import requests

save_dir = 'C:\\Users\\chris\Documents\\personal_projects'
os.chdir(save_dir)

conn = sqlite3.connect("biotech.db")

df = pd.read_sql_query("""
    WITH latest_trial AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY snapshot_date DESC) AS rn
        FROM trials
    ),
    latest_pub AS (
        SELECT nct_id, title AS latest_publication, pub_date,
               ROW_NUMBER() OVER (PARTITION BY nct_id ORDER BY pub_date DESC) AS rn
        FROM publications
    )
    SELECT t.company, t.nct_id, t.phase, t.status, t.conditions,
           t.primary_completion_date, t.enrollment,
           p.latest_publication, p.pub_date
    FROM latest_trial t
    LEFT JOIN latest_pub p
        ON t.nct_id = p.nct_id AND p.rn = 1
    WHERE t.rn = 1
      AND t.phase IN ('PHASE3', 'PHASE4')
      AND t.status = 'ACTIVE_NOT_RECRUITING'
      AND t.primary_completion_date BETWEEN date('now') AND date('now', '+6 months')
    ORDER BY t.primary_completion_date
""", conn)

print(len(df))
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
df.sort_values('primary_completion_date')[['company', 'phase', 'primary_completion_date', 'conditions', 'enrollment', 'nct_id', 'latest_publication']].head(40)
#pd.reset_option('display.max_colwidth')
#pd.reset_option('display.max_rows')

136


,company,phase,primary_completion_date,conditions,enrollment,nct_id,latest_publication
0,sanofi,PHASE3,2026-06-08,Lichen Simplex Chronicus,142,NCT06687967,None
1,sanofi,PHASE3,2026-06-08,Lichen Simplex Chronicus,138,NCT06687980,None
2,novo nordisk,PHASE3,2026-06-09,"Cardiovascular Risk,Chronic Kidney Disease,Inflammation",6200,NCT05021835,Novel Therapeutic Strategies for Patients With CKD and Diabetes Mellitus.
3,gsk,PHASE3,2026-06-11,"Cough,Refractory Chronic Cough",975,NCT05600777,"Mepolizumab for the treatment of refractory chronic cough in patients with eosinophilic airways disease (MUCOSA): a randomised, double-blind, parallel-group, placebo-controlled trial."
4,astrazeneca,PHASE3,2026-06-15,"Gastric Cancer,Gastroesophageal Junction Cancer",592,NCT06346392,Antibodies to watch in 2026.
5,astrazeneca,PHASE3,2026-06-18,"Carcinoma,Non-Small-Cell Lung",345,NCT05261399,Targeting MET in non-small cell lung cancer brain metastases: from molecular mechanism to precision therapy.
6,amgen,PHASE3,2026-06-22,Atopic Dermatitis,2621,NCT05882877,The Role of OX40 Pathway Inhibition as a New Therapeutic Strategy for Atopic Dermatitis.
7,sanofi,PHASE4,2026-06-26,Type 2 Diabetes,105,NCT06716424,None
9,novo nordisk,PHASE3,2026-06-29,Type 2 Diabetes,264,NCT07271251,None
8,sanofi,PHASE3,2026-06-29,Dermatitis Atopic,1541,NCT06407934,The Role of OX40 Pathway Inhibition as a New Therapeutic Strategy for Atopic Dermatitis.
